# KnowWhenToHandOff: a tour of the data

A tool-using customer-service agent (Qwen3-4B on τ²-bench airline and retail)
has to decide when to **hand off to a human**. This dataset holds the hand-off
task set and the records of every run: prompted baselines, SFT, and two GRPO
variants (R1: task reward only, R2: Responsible-AI reward).

This notebook needs no GPU. It walks through:

1. the run index and headline metrics,
2. the task set and its should-transfer labels,
3. evaluation conversations, and where each model transfers,
4. the GRPO training rollouts, where R1 learns to over-escalate.

Code: [github.com/JaspinXu/KnowWhenToHandOff](https://github.com/JaspinXu/KnowWhenToHandOff) ·
Models and primary copy of this dataset: [huggingface.co/JaspinXu](https://huggingface.co/JaspinXu)

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_colwidth", 80)


def find_root() -> Path:
    """The dataset folder: $HANDOFF_RELEASE, or wherever Kaggle mounted it."""
    if os.environ.get("HANDOFF_RELEASE"):
        return Path(os.environ["HANDOFF_RELEASE"])
    for index in sorted(Path("/kaggle/input").rglob("runs.jsonl")):
        return index.parent
    raise FileNotFoundError("Attach the KnowWhenToHandOff dataset (Add Input).")


root = find_root()
print(root, sum(1 for p in root.rglob("*") if p.is_file()), "files")

## 1. Run index

`runs.jsonl` has one row per run. Run IDs start with the model (`B0` prompted
Qwen3-4B, `B1` prompted Qwen3-30B-A3B, `B2` SFT, `R1` and `R2` GRPO), then
name the protocol (`test_main`, `test_styles`, `test_handoff`, `dev`).
Over-escalation is the share of conversations that should **not** be
transferred but were.

In [ ]:
runs = pd.read_json(root / "runs.jsonl", lines=True)
metrics = [m for m in ["success", "handoff_f1", "over_escalation_rate"] if m in runs]
tests = runs[runs.run_id.str.contains("-test_")].sort_values("run_id")
tests[["run_id", *metrics]].reset_index(drop=True)

## 2. Tasks and labels

The derived tasks come in three kinds: *explicit request* and *out of scope*
(the agent should transfer) and *hard negative* (a frustrated or hurried user
whose issue the agent can solve, so it should not). The manifest lists each
task's split, kind and label; the task files use τ²-bench's own schema.

In [ ]:
manifest = json.loads((root / "tasks/manifests/handoff_tasks_v1.json").read_text())
tasks = pd.DataFrame(
    [
        {**task, "domain": domain}
        for domain, entry in manifest["domains"].items()
        for task in entry["tasks"]
    ]
)
pd.crosstab([tasks.domain, tasks.split], tasks.kind, margins=True)

In [ ]:
airline = json.loads((root / "tasks/v1/airline_tasks.json").read_text())
example = next(t for t in airline if t["id"].endswith("hn0"))
print(example["id"], "|", example["description"]["purpose"])
print(example["user_scenario"]["instructions"]["reason_for_call"])

## 3. Evaluation conversations

Each line of `runs/<run id>/trajectories.jsonl` is one conversation with its
messages, tool calls, rule violations, task reward and whether (and when) the
agent transferred. Below: the transfer rate per task kind on the hand-off test
set for SFT (B2), R1 and R2. A good agent transfers on the first two kinds and
not on hard negatives.

In [ ]:
def trajectories(model: str, protocol: str = "test_handoff") -> pd.DataFrame:
    """All conversations of one model under one evaluation protocol."""
    ids = [r for r in runs.run_id if r.startswith(f"{model}-") and protocol in r]
    frames = [
        pd.read_json(root / "runs" / run_id / "trajectories.jsonl", lines=True)
        for run_id in ids
    ]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


evals = {m: trajectories(m) for m in ["B2", "R1", "R2"]}
pd.DataFrame(
    {m: df.groupby("kind").transferred.mean() for m, df in evals.items() if len(df)}
).round(3)

In [ ]:
def show(conversation: pd.Series, limit: int = 12) -> None:
    """Print the first messages of a conversation, one line each."""
    print(
        conversation.task_id,
        conversation.kind,
        "transferred:",
        conversation.transferred,
    )
    for message in conversation.messages[:limit]:
        text = message.get("content") or ""
        calls = [
            c.get("name") or c.get("function", {}).get("name")
            for c in message.get("tool_calls") or []
        ]
        if calls:
            text = f"{text} [tool calls: {', '.join(map(str, calls))}]".strip()
        print(f"{message.get('role', '?'):>9}: {text[:160]}")


r1 = evals["R1"]
over = r1[(r1.kind == "hard_negative") & r1.transferred]
show(over.iloc[0] if len(over) else r1.iloc[0])

## 4. Training rollouts: the loophole opening

`runs/R1-s1-*/rollouts/` and `runs/R2-s1-*/rollouts/` hold every GRPO training
episode (60 steps × 64 episodes) with its reward terms. Under the task-only
reward a transfer scores 1.0 whenever the reference solution changes nothing,
so "transfer when unsure" becomes weakly dominant. The plot tracks the transfer
rate on episodes that should **not** be transferred.

In [ ]:
def rollouts(model: str) -> pd.DataFrame:
    """Training episodes of one GRPO run in time order, with a step number."""
    run = next(r for r in runs.run_id if r.startswith(f"{model}-s1-"))
    parts = sorted((root / "runs" / run / "rollouts").glob("*.jsonl"))
    df = pd.concat([pd.read_json(p, lines=True) for p in parts], ignore_index=True)
    df = df[~df.validate.astype(bool)].sort_values("time", ignore_index=True)
    df["step"] = df.index // 64 + 1  # 8 tasks x 8 rollouts per step
    if "infra_failure" in df:  # episodes lost to infrastructure carry no signal
        df = df[df.infra_failure != 1]
    return df


colours = {"R1": "#EA580C", "R2": "#059669"}
fig, ax = plt.subplots(figsize=(7, 3.6))
for model, colour in colours.items():
    df = rollouts(model)
    negatives = df[df.should_transfer == 0]
    rate = negatives.groupby("step").transferred.mean().rolling(5, min_periods=1).mean()
    ax.plot(rate.index, rate.values, color=colour, linewidth=2, label=model)
ax.set_xlabel("GRPO step")
ax.set_ylabel("transfer rate, should-not-transfer")
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.show()

## Next

* Model weights: `JaspinXu/KnowWhenToHandOff-Qwen3-4B-R2` (recommended), `-R1`
  and `-SFT` on Hugging Face, each with a vLLM quick start.
* Rebuild every number in the report from these files with the scripts in the
  code repository; `provenance.json` in each run pins the code and model
  revisions.
* Licence: MIT. The tasks derive from τ²-bench (MIT, © 2025 Sierra Research);
  keep its notice when redistributing. Research use only.